In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.ensemble import IsolationForest
from sklearn.cluster import DBSCAN
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.metrics import average_precision_score,precision_recall_curve

In [ ]:
#Load dataset
df = pd.read_csv("/content/creditcard.csv.zip")
df.head()

In [ ]:
df.shape

In [ ]:
#Check class distribution
df['Class'].value_counts()

In [ ]:
#for Isolation forest we use the entire dataset

In [ ]:
X_full = df.drop("Class", axis=1)
y_full = df["Class"]

In [ ]:
X_train_full, X_val_full, y_train_full, y_val_full = train_test_split(
    X_full, y_full,
    test_size=0.3,
    stratify=y_full,
    random_state=42
)

In [ ]:
scaler_full = StandardScaler()

X_train_full = scaler_full.fit_transform(X_train_full)
X_val_full = scaler_full.transform(X_val_full)

In [ ]:
#for LOF and DBSCAN we don't use the entire dataset, instead use a sample data of 50000 rows

In [ ]:
df_sample = df.sample(50000, random_state=42)

In [ ]:
X_sample = df_sample.drop("Class", axis=1)
y_sample = df_sample["Class"]

In [ ]:
X_train_s, X_val_s, y_train_s, y_val_s = train_test_split(
    X_sample, y_sample,
    test_size=0.3,
    stratify=y_sample,
    random_state=42
)

In [ ]:
scaler_sample = StandardScaler()

X_train_s = scaler_sample.fit_transform(X_train_s)
X_val_s = scaler_sample.transform(X_val_s)

In [ ]:
pca = PCA(n_components=8)

X_train_pca = pca.fit_transform(X_train_s)
X_val_pca = pca.transform(X_val_s)

# **Isolation Forest**

Model quality → PR-AUC

Decision policy → F1

In [ ]:
#Hyperparameter tuning Using PR-AUC
best_ap = 0
best_model = None

for n in [100, 200]:
    for max_s in [0.7, 1.0]:
        iso = IsolationForest(
            n_estimators=n,
            max_samples=max_s,
            contamination=0.01,   #proportion of dataset that is expected to be anomalous / to define threshold on the anomaly scores
            random_state=42)

        iso.fit(X_train_full)

        scores = -iso.decision_function(X_val_full)
        #Isolation Forest decision_function returns higher values for normal observations.
        #Since PR-AUC expects higher scores to indicate the positive class (fraud),
        #We invert the sign to align anomaly severity with the positive label.

        ap = average_precision_score(y_val_full, scores)
        #This summarizes the entire PR curve into one number.

        if ap > best_ap:
            best_ap = ap
            best_model = iso

print("Best AP:", best_ap)

In [ ]:
#Choose Best Threshold Using F1
scores_val = -best_model.decision_function(X_val_full) # Isolation Forest scores

thresholds = np.linspace(scores_val.min(), scores_val.max(), 100)

best_f1 = 0
best_thresh = None

for t in thresholds:
    preds = (scores_val > t).astype(int)
    f1 = f1_score(y_val_full, preds)

    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print("Best F1:", best_f1)
print("Best Threshold:", best_thresh)

In [ ]:
# Isolation Forest scores
scores_iso = -best_model.decision_function(X_val_full)

# Use best threshold which was found earlier
preds_iso = (scores_iso > best_thresh).astype(int)

#for PR curve
precision_curve_iso, recall_curve_iso, _ = precision_recall_curve(y_val_full, scores_iso)
#This uses scores, not predictions.So it evaluates performance across all possible thresholds.

#for metrics
precision_iso = precision_score(y_val_full, preds_iso)
recall_iso = recall_score(y_val_full, preds_iso)
f1_iso = f1_score(y_val_full, preds_iso)
ap_iso = average_precision_score(y_val_full, scores_iso)
#This summarizes the entire PR curve into one number.

print(classification_report(y_val_full, preds_iso))

# **LOF**

In [ ]:
#Hyperparameter tuning Using PR-AUC
best_ap_lof = 0
best_lof = None

for n in [20, 50, 100]:
    lof = LocalOutlierFactor(
        n_neighbors=n,
        contamination=0.01,
        novelty=True)

    lof.fit(X_train_pca)

    scores = -lof.decision_function(X_val_pca)
    ap = average_precision_score(y_val_s, scores)

    if ap > best_ap_lof:
        best_ap_lof = ap
        best_lof = lof

print("Best LOF AP:", best_ap_lof)

In [ ]:
scores_lof = -best_lof.decision_function(X_val_pca)

thresholds = np.linspace(scores_lof.min(), scores_lof.max(), 100)

best_f1_lof = 0
best_thresh_lof = None

for t in thresholds:
    preds = (scores_lof > t).astype(int)
    f1 = f1_score(y_val_s, preds)

    if f1 > best_f1_lof:
        best_f1_lof = f1
        best_thresh_lof = t

print("Best LOF F1:", best_f1_lof)
print("Best Threshold:", best_thresh_lof)

In [ ]:
scores_lof = -best_lof.decision_function(X_val_pca)
preds_lof = (scores_lof > best_thresh_lof).astype(int)
scores_lof = -best_lof.decision_function(X_val_pca)

#for PR curve
precision_curve_lof, recall_curve_lof, _ = precision_recall_curve(y_val_s, scores_lof)
#for final metrics
precision_lof = precision_score(y_val_s, preds_lof)
recall_lof = recall_score(y_val_s, preds_lof)
f1_lof = f1_score(y_val_s,preds_lof)
ap_lof = average_precision_score(y_val_s,preds_lof)

In [ ]:
plt.figure(figsize=(7,6))

plt.plot(recall_curve_iso, precision_curve_iso,
         label=f'Isolation Forest (AP={ap_iso:.3f})')

plt.plot(recall_curve_lof, precision_curve_lof,
         label=f'LOF (AP={ap_lof:.3f})')

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve Comparison")
plt.legend()
plt.grid(True)

plt.show()

Interpretation:
Isolation Forest consistently maintains higher precision across recall levels compared to LOF. The higher PR-AUC indicates better ranking of fraudulent transactions. LOF struggles in high-dimensional space.

# **DBSCAN**

In [ ]:
#DBSCAN does NOT produce anomaly scores.
#So, we cannot compute PR-AUC properly.
#We can only tune using F1 on binary predictions.

In [ ]:
best_f1_db = 0
best_params = None

for eps in [0.8, 1.0, 1.2]:
    for ms in [5, 10]:

        db = DBSCAN(eps=eps, min_samples=ms)
        preds = db.fit_predict(X_val_pca)  #DBSCAN does not support .predict() so it runs on same data we evaluate

        preds = np.where(preds == -1, 1, 0)

        f1 = f1_score(y_val_s, preds)

        if f1 > best_f1_db:
            best_f1_db = f1
            best_params = (eps, ms)

print("Best DBSCAN F1:", best_f1_db)
print("Best Params:", best_params)


In [ ]:
eps, ms = best_params

db = DBSCAN(eps=eps, min_samples=ms)

preds_db = db.fit_predict(X_val_pca)
preds_db = np.where(preds_db == -1, 1, 0)

precision_db = precision_score(y_val_s, preds_db)
recall_db = recall_score(y_val_s, preds_db)
f1_db = f1_score(y_val_s, preds_db)

In [ ]:
print(confusion_matrix(y_val_s, preds))
print(classification_report(y_val_s, preds))

In [ ]:
import pandas as pd
import numpy as np

results = pd.DataFrame({
    "Model": ["Isolation Forest", "LOF", "DBSCAN"],
    "Precision": [precision_iso, precision_lof, precision_db],
    "Recall": [recall_iso, recall_lof, recall_db],
    "F1 Score": [f1_iso, f1_lof, f1_db],
    "PR-AUC": [ap_iso, ap_lof, np.nan]  # DBSCAN no PR-AUC
})

results

Isolation Forest achieved the highest PR-AUC and maintained a balanced precision-recall trade-off. Density-based methods like LOF and DBSCAN struggled due to high dimensionality and sparsity effects.”